# 27 · Save, Load & Reuse a Model

**2026 learning module** · Run cells from top to bottom. This notebook uses built-in/sample datasets and fixed random seeds where appropriate.

## Goal
Save the complete preprocessing/model pipeline with `joblib`, reload it, and verify predictions. Only load model files you trust: pickle/joblib formats can execute code during loading.

In [ ]:
from pathlib import Path
import json
import joblib
import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
data = load_iris(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, stratify=data.target, random_state=42)
pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
pipeline.fit(X_train, y_train)
pred_before = pipeline.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, pred_before), 3))

In [ ]:
model_path = artifact_dir / "iris_pipeline.joblib"
metadata_path = artifact_dir / "iris_pipeline_metadata.json"
joblib.dump(pipeline, model_path)
metadata = {
    "model_name": "iris-standard-scaled-logistic-regression",
    "python_note": "Record your Python version in your own deployment environment",
    "scikit_learn_version": sklearn.__version__,
    "features": list(X_train.columns),
    "target_names": list(data.target_names),
    "test_accuracy": float(accuracy_score(y_test, pred_before)),
    "random_state": 42,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved:", model_path.resolve())
print("Saved:", metadata_path.resolve())

In [ ]:
reloaded = joblib.load(model_path)  # Only load files you created or otherwise trust.
pred_after = reloaded.predict(X_test)
assert (pred_before == pred_after).all(), "Predictions changed after reload"
print("Reload verification passed; predictions are identical.")
print(json.loads(metadata_path.read_text(encoding="utf-8")))

## Production reminders
- Keep the preprocessing steps inside the saved pipeline.
- Pin and record dependency versions for reproducibility.
- Validate request inputs and add monitoring before exposing a model as a service.
- Never load untrusted `.joblib` or pickle files.